In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

# -----------------------------------
# Load MNIST test dataset
# -----------------------------------
transform = transforms.ToTensor()

mnist_test = datasets.MNIST(
    root="./classical_data",
    train=False,
    download=True,
    transform=transform
)

# -----------------------------------
# Split test data into 10 equal parts
# -----------------------------------
num_parts = 10
part_size = len(mnist_test) // num_parts

lengths = [part_size] * num_parts

generator = torch.Generator().manual_seed(42)

test_parts = random_split(
    mnist_test,
    lengths,
    generator=generator
)

# -----------------------------------
# Create DataLoaders for each part
# -----------------------------------
test_loaders = [
    DataLoader(
        part,
        batch_size=64,
        shuffle=False
    )
    for part in test_parts
]

# -----------------------------------
# Check sizes
# -----------------------------------
for i, part in enumerate(test_parts):
    print(f"Test Part {i + 1}: {len(part)} samples")



100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.82MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.51MB/s]

Test Part 1: 1000 samples
Test Part 2: 1000 samples
Test Part 3: 1000 samples
Test Part 4: 1000 samples
Test Part 5: 1000 samples
Test Part 6: 1000 samples
Test Part 7: 1000 samples
Test Part 8: 1000 samples
Test Part 9: 1000 samples
Test Part 10: 1000 samples


In [2]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# -----------------------------------
# Load MNIST test dataset
# -----------------------------------
transform = transforms.ToTensor()

mnist_test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# -----------------------------------
# Split into 10 equal parts
# -----------------------------------
num_parts = 10
part_size = len(mnist_test) // num_parts
lengths = [part_size] * num_parts

generator = torch.Generator().manual_seed(42)

test_parts = random_split(
    mnist_test,
    lengths,
    generator=generator
)

# -----------------------------------
# Folder to save splits
# -----------------------------------
save_dir = "./mnist_test_splits"
os.makedirs(save_dir, exist_ok=True)

# -----------------------------------
# Save each split
# -----------------------------------
for i, subset in enumerate(test_parts):

    images = []
    labels = []

    for image, label in subset:
        images.append(image)
        labels.append(label)

    images = torch.stack(images)
    labels = torch.tensor(labels)

    save_path = os.path.join(
        save_dir,
        f"mnist_test_part_{i+1}.pt"
    )

    torch.save(
        {
            "images": images,
            "labels": labels
        },
        save_path
    )

    print(
        f"Saved Part {i+1}: "
        f"{len(labels)} samples -> {save_path}"
    )

Saved Part 1: 1000 samples -> ./mnist_test_splits\mnist_test_part_1.pt
Saved Part 2: 1000 samples -> ./mnist_test_splits\mnist_test_part_2.pt
Saved Part 3: 1000 samples -> ./mnist_test_splits\mnist_test_part_3.pt
Saved Part 4: 1000 samples -> ./mnist_test_splits\mnist_test_part_4.pt
Saved Part 5: 1000 samples -> ./mnist_test_splits\mnist_test_part_5.pt
Saved Part 6: 1000 samples -> ./mnist_test_splits\mnist_test_part_6.pt
Saved Part 7: 1000 samples -> ./mnist_test_splits\mnist_test_part_7.pt
Saved Part 8: 1000 samples -> ./mnist_test_splits\mnist_test_part_8.pt
Saved Part 9: 1000 samples -> ./mnist_test_splits\mnist_test_part_9.pt
Saved Part 10: 1000 samples -> ./mnist_test_splits\mnist_test_part_10.pt


In [3]:
data = torch.load(
    "./mnist_test_splits/mnist_test_part_1.pt"
)

images = data["images"]
labels = data["labels"]

print(images.shape)
print(labels.shape)

torch.Size([1000, 1, 28, 28])
torch.Size([1000])


In [4]:
import os
import torch

save_dir = "./mnist_test_splits"

all_splits = []

for i in range(1, 11):
    file_path = os.path.join(
        save_dir,
        f"mnist_test_part_{i}.pt"
    )

    data = torch.load(file_path)

    images = data["images"]
    labels = data["labels"]

    all_splits.append({
        "images": images,
        "labels": labels
    })

    print(
        f"Part {i}: "
        f"images={images.shape}, "
        f"labels={labels.shape}"
    )

Part 1: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 2: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 3: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 4: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 5: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 6: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 7: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 8: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 9: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])
Part 10: images=torch.Size([1000, 1, 28, 28]), labels=torch.Size([1000])


In [6]:
import torch

data = torch.load(
    "./mnist_test_splits/mnist_test_part_1.pt"
)

images = data["images"]
labels = data["labels"]

epsilon = 0.1

noise = torch.empty_like(images).uniform_(
    -epsilon,
    epsilon
)

perturbed_images = torch.clamp(
    images + noise,
    0.0,
    1.0
)

torch.save(
    {
        "images": perturbed_images,
        "labels": labels,
        "epsilon": epsilon
    },
    "./mnist_test_splits/mnist_test_part_1_random_eps_0.1.pt"
)

print("Saved perturbed dataset.")

Saved perturbed dataset.


In [7]:
for i in range(1, 11):
    
    data = torch.load(
        f"./mnist_test_splits/mnist_test_part_{i}.pt"
    )

    images = data["images"]
    labels = data["labels"]

    epsilon = i/10

    noise = torch.empty_like(images).uniform_(
        -epsilon,
        epsilon
    )

    perturbed_images = torch.clamp(
        images + noise,
        0.0,
        1.0
    )

    torch.save(
        {
            "images": perturbed_images,
            "labels": labels,
            "epsilon": epsilon
        },
        f"./mnist_test_splits/mnist_test_part_{i}_random_eps_{epsilon}.pt"
    )

    print("Saved perturbed dataset.")

Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.
Saved perturbed dataset.


In [1]:
for i in range(1, 11):
    
    data = torch.load(
        f"./mnist_test_splits/mnist_test_part_{i}.pt"
    )

NameError: name 'torch' is not defined

# cifar10

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

# -----------------------------------
# Load CIFAR-10 test dataset
# -----------------------------------
transform = transforms.ToTensor()

cifar10_test = datasets.CIFAR10(
    root="./cifar10_data",
    train=False,
    download=True,
    transform=transform
)

# -----------------------------------
# Split test data into 10 equal parts
# -----------------------------------
num_parts = 10
part_size = len(cifar10_test) // num_parts

lengths = [part_size] * num_parts

generator = torch.Generator().manual_seed(42)

test_parts = random_split(
    cifar10_test,
    lengths,
    generator=generator
)

# -----------------------------------
# Create DataLoaders for each part
# -----------------------------------
test_loaders = [
    DataLoader(
        part,
        batch_size=64,
        shuffle=False
    )
    for part in test_parts
]

# -----------------------------------
# Check sizes
# -----------------------------------
for i, part in enumerate(test_parts):
    print(f"Test Part {i + 1}: {len(part)} samples")

Test Part 1: 1000 samples
Test Part 2: 1000 samples
Test Part 3: 1000 samples
Test Part 4: 1000 samples
Test Part 5: 1000 samples
Test Part 6: 1000 samples
Test Part 7: 1000 samples
Test Part 8: 1000 samples
Test Part 9: 1000 samples
Test Part 10: 1000 samples


In [5]:
import os
import torch

from torchvision import datasets, transforms
from torch.utils.data import random_split


# -----------------------------------
# Load CIFAR-10 test dataset
# -----------------------------------

transform = transforms.ToTensor()

cifar10_test = datasets.CIFAR10(
    root="./cifar10_data",
    train=False,
    download=True,
    transform=transform
)


# -----------------------------------
# Split into 10 equal parts
# -----------------------------------

num_parts = 10

part_size = len(cifar10_test) // num_parts

lengths = [part_size] * num_parts

generator = torch.Generator().manual_seed(42)

test_parts = random_split(
    cifar10_test,
    lengths,
    generator=generator
)


# -----------------------------------
# Folder to save splits
# -----------------------------------

save_dir = "./cifar10_test_splits"

os.makedirs(
    save_dir,
    exist_ok=True
)


# -----------------------------------
# Save each split
# -----------------------------------

for i, subset in enumerate(test_parts):

    images = []
    labels = []

    for image, label in subset:

        images.append(image)
        labels.append(label)


    # CIFAR-10 image shape:
    # [3, 32, 32]
    #
    # After stacking 1000 images:
    # [1000, 3, 32, 32]

    images = torch.stack(images)

    labels = torch.tensor(
        labels,
        dtype=torch.long
    )


    save_path = os.path.join(
        save_dir,
        f"cifar10_test_part_{i + 1}.pt"
    )


    torch.save(
        {
            "images": images,
            "labels": labels
        },
        save_path
    )


    print(
        f"Saved Part {i + 1}: "
        f"{len(labels)} samples -> "
        f"{save_path}"
    )

    print(
        f"  Images shape: {images.shape}"
    )

    print(
        f"  Labels shape: {labels.shape}"
    )

Saved Part 1: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_1.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 2: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_2.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 3: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_3.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 4: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_4.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 5: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_5.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 6: 1000 samples -> ./cifar10_test_splits\cifar10_test_part_6.pt
  Images shape: torch.Size([1000, 3, 32, 32])
  Labels shape: torch.Size([1000])
Saved Part 7: 1000 samples -> ./cifar10_test_splits\cifar10_test

In [6]:
import os
import torch


# -----------------------------------
# Folder containing CIFAR-10 splits
# -----------------------------------

data_dir = "./cifar10_test_splits"


# -----------------------------------
# Add random perturbation
# -----------------------------------

for i in range(1, 11):

    data = torch.load(
        f"{data_dir}/cifar10_test_part_{i}.pt"
    )

    images = data["images"]
    labels = data["labels"]


    # -----------------------------------
    # Epsilon
    # -----------------------------------

    epsilon = i / 10


    # -----------------------------------
    # Random uniform perturbation
    # -----------------------------------

    noise = torch.empty_like(
        images
    ).uniform_(
        -epsilon,
        epsilon
    )


    # -----------------------------------
    # Add perturbation
    # -----------------------------------

    perturbed_images = torch.clamp(
        images + noise,
        0.0,
        1.0
    )


    # -----------------------------------
    # Save perturbed CIFAR-10 split
    # -----------------------------------

    save_path = (
        f"{data_dir}/"
        f"cifar10_test_part_{i}_"
        f"random_eps_{epsilon:.1f}.pt"
    )


    torch.save(
        {
            "images": perturbed_images,
            "labels": labels,
            "epsilon": epsilon
        },
        save_path
    )


    print(
        f"Saved Part {i}: "
        f"epsilon={epsilon:.1f} -> "
        f"{save_path}"
    )

Saved Part 1: epsilon=0.1 -> ./cifar10_test_splits/cifar10_test_part_1_random_eps_0.1.pt
Saved Part 2: epsilon=0.2 -> ./cifar10_test_splits/cifar10_test_part_2_random_eps_0.2.pt
Saved Part 3: epsilon=0.3 -> ./cifar10_test_splits/cifar10_test_part_3_random_eps_0.3.pt
Saved Part 4: epsilon=0.4 -> ./cifar10_test_splits/cifar10_test_part_4_random_eps_0.4.pt
Saved Part 5: epsilon=0.5 -> ./cifar10_test_splits/cifar10_test_part_5_random_eps_0.5.pt
Saved Part 6: epsilon=0.6 -> ./cifar10_test_splits/cifar10_test_part_6_random_eps_0.6.pt
Saved Part 7: epsilon=0.7 -> ./cifar10_test_splits/cifar10_test_part_7_random_eps_0.7.pt
Saved Part 8: epsilon=0.8 -> ./cifar10_test_splits/cifar10_test_part_8_random_eps_0.8.pt
Saved Part 9: epsilon=0.9 -> ./cifar10_test_splits/cifar10_test_part_9_random_eps_0.9.pt
Saved Part 10: epsilon=1.0 -> ./cifar10_test_splits/cifar10_test_part_10_random_eps_1.0.pt


In [7]:
import os
import torch


# ============================================================
# Configuration
# ============================================================

DATA_DIR = "./cifar10_test_splits"

OUTPUT_DIR = "./cifar10_adversarial_splits"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# Generate perturbed CIFAR-10 datasets
# ============================================================

for i in range(1, 11):

    # --------------------------------------------------------
    # Load original CIFAR-10 split
    # --------------------------------------------------------

    input_path = os.path.join(
        DATA_DIR,
        f"cifar10_test_part_{i}.pt"
    )

    data = torch.load(
        input_path,
        map_location="cpu"
    )

    images = data["images"]
    labels = data["labels"]


    # --------------------------------------------------------
    # Epsilon: 0.1 -> 1.0
    # --------------------------------------------------------

    epsilon = i / 10.0


    print(
        f"\nProcessing Part {i}"
    )

    print(
        f"Epsilon: {epsilon:.1f}"
    )

    print(
        "Original image shape:",
        images.shape
    )


    # --------------------------------------------------------
    # Random uniform perturbation
    #
    # noise ~ Uniform(-epsilon, +epsilon)
    # --------------------------------------------------------

    noise = torch.empty_like(
        images
    ).uniform_(
        -epsilon,
        epsilon
    )


    # --------------------------------------------------------
    # Add perturbation
    # --------------------------------------------------------

    perturbed_images = (
        images
        +
        noise
    )


    # --------------------------------------------------------
    # Keep CIFAR-10 pixels inside [0, 1]
    # --------------------------------------------------------

    perturbed_images = torch.clamp(
        perturbed_images,
        min=0.0,
        max=1.0
    )


    # --------------------------------------------------------
    # Output path
    # --------------------------------------------------------

    output_path = os.path.join(
        OUTPUT_DIR,
        f"cifar10_test_part_{i}_random_eps_{epsilon:.1f}.pt"
    )


    # --------------------------------------------------------
    # Save perturbed dataset
    # --------------------------------------------------------

    torch.save(
        {
            "images": perturbed_images,
            "labels": labels,
            "epsilon": epsilon,
            "noise": noise
        },
        output_path
    )


    # --------------------------------------------------------
    # Perturbation statistics
    # --------------------------------------------------------

    actual_difference = (
        perturbed_images
        -
        images
    )

    max_difference = (
        actual_difference
        .abs()
        .max()
        .item()
    )

    mean_difference = (
        actual_difference
        .abs()
        .mean()
        .item()
    )


    print(
        f"Samples: {len(labels)}"
    )

    print(
        f"Max |perturbation|: "
        f"{max_difference:.6f}"
    )

    print(
        f"Mean |perturbation|: "
        f"{mean_difference:.6f}"
    )

    print(
        f"Saved -> {output_path}"
    )


print(
    "\nAll CIFAR-10 adversarial "
    "datasets generated successfully."
)


Processing Part 1
Epsilon: 0.1
Original image shape: torch.Size([1000, 3, 32, 32])
Samples: 1000
Max |perturbation|: 0.100000
Mean |perturbation|: 0.048818
Saved -> ./cifar10_adversarial_splits\cifar10_test_part_1_random_eps_0.1.pt

Processing Part 2
Epsilon: 0.2
Original image shape: torch.Size([1000, 3, 32, 32])
Samples: 1000
Max |perturbation|: 0.200000
Mean |perturbation|: 0.095738
Saved -> ./cifar10_adversarial_splits\cifar10_test_part_2_random_eps_0.2.pt

Processing Part 3
Epsilon: 0.3
Original image shape: torch.Size([1000, 3, 32, 32])
Samples: 1000
Max |perturbation|: 0.300000
Mean |perturbation|: 0.139908
Saved -> ./cifar10_adversarial_splits\cifar10_test_part_3_random_eps_0.3.pt

Processing Part 4
Epsilon: 0.4
Original image shape: torch.Size([1000, 3, 32, 32])
Samples: 1000
Max |perturbation|: 0.400000
Mean |perturbation|: 0.180846
Saved -> ./cifar10_adversarial_splits\cifar10_test_part_4_random_eps_0.4.pt

Processing Part 5
Epsilon: 0.5
Original image shape: torch.Size([10

In [8]:
data = torch.load(
    "./cifar10_adversarial_splits/"
    "cifar10_test_part_5_random_eps_0.5.pt"
)

images = data["images"]
labels = data["labels"]
epsilon = data["epsilon"]

print(images.shape)
print(labels.shape)
print(epsilon)

torch.Size([1000, 3, 32, 32])
torch.Size([1000])
0.5


# MNIST, FashionMnist, KMnist

In [3]:
"""
Multi-Dataset Adversarial Generation (FGSM): MNIST, FashionMNIST, KMNIST
================================================================================

What this script does, for EACH of MNIST, FashionMNIST, and Kuzushiji-MNIST:
1. Loads that dataset's test set.
2. Splits it into 10 equal parts and saves each part to disk.
3. Generates FGSM (Fast Gradient Sign Method) adversarial examples against
   that dataset's own trained CNN checkpoint, using an increasing epsilon
   per part (part 1 -> eps=0.1, part 10 -> eps=1.0).
4. Saves each perturbed part alongside the original, labeled by epsilon.
5. Reports clean vs. adversarial accuracy per epsilon.

Each dataset gets its own splits folder and its own checkpoint, so results
never mix across datasets:
    MNIST           -> ./mnist_test_splits/            + mnist_cnn.pth
    FashionMNIST     -> ./fashionmnist_test_splits/     + fashionmnist_cnn.pth
    KMNIST           -> ./kmnist_test_splits/           + kmnist_cnn.pth

FGSM vs. random noise:
    Uniform random noise perturbs every pixel in a random direction and
    ignores the model entirely. FGSM instead perturbs each pixel in the
    direction that INCREASES the model's loss (using the sign of the
    gradient of the loss w.r.t. the input), which is what actually degrades
    accuracy in a targeted way and is the standard baseline for adversarial
    robustness testing.

Requires trained checkpoints ("mnist_cnn.pth", "fashionmnist_cnn.pth",
"kmnist_cnn.pth") matching the SimpleCNN architecture below.

Install once:
pip install torch torchvision tqdm
"""

from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

# =============================================================================
# CONFIGURATION
# =============================================================================

SEED = 42
NUM_PARTS = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "./classical_data"

# Per-dataset configuration: dataset class, normalization stats (matching
# training), checkpoint filename, and its own splits directory.
DATASET_CONFIGS = {
    "MNIST": {
        "dataset_class": datasets.MNIST,
        "mean": (0.1307,),
        "std": (0.3081,),
        "checkpoint_path": "mnist_cnn.pth",
        "splits_dir": Path("./mnist_test_splits"),
    },
    "FashionMNIST": {
        "dataset_class": datasets.FashionMNIST,
        "mean": (0.2860,),
        "std": (0.3530,),
        "checkpoint_path": "fashionmnist_cnn.pth",
        "splits_dir": Path("./fashionmnist_test_splits"),
    },
    "KMNIST": {
        "dataset_class": datasets.KMNIST,  # Kuzushiji-MNIST
        "mean": (0.1918,),
        "std": (0.3483,),
        "checkpoint_path": "kmnist_cnn.pth",
        "splits_dir": Path("./kmnist_test_splits"),
    },
}

for config in DATASET_CONFIGS.values():
    config["splits_dir"].mkdir(parents=True, exist_ok=True)


# =============================================================================
# MODEL (must match the architecture used to produce each *_cnn.pth)
# =============================================================================

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def load_model(checkpoint_path: str) -> SimpleCNN:
    model = SimpleCNN().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    model.eval()
    return model


# =============================================================================
# DATASET SPLITTING
# =============================================================================

def load_and_split_test_set(dataset_class, dataset_name: str) -> list:
    """Loads a dataset's test set and splits it into NUM_PARTS equal parts."""
    transform = transforms.ToTensor()
    test_dataset = dataset_class(
        root=DATA_ROOT,
        train=False,
        download=True,
        transform=transform,
    )

    part_size = len(test_dataset) // NUM_PARTS
    remainder = len(test_dataset) - part_size * NUM_PARTS
    # Distribute any remainder samples across the first few parts so no
    # data is silently dropped (random_split requires lengths to sum
    # exactly to len(dataset)).
    lengths = [part_size + (1 if i < remainder else 0) for i in range(NUM_PARTS)]

    generator = torch.Generator().manual_seed(SEED)
    test_parts = random_split(test_dataset, lengths, generator=generator)

    print(f"\n{dataset_name}: {len(test_dataset):,} total test samples")
    for i, part in enumerate(test_parts):
        print(f"  Part {i + 1}: {len(part)} samples")

    return test_parts


def save_splits_to_disk(test_parts: list, splits_dir: Path, prefix: str) -> None:
    """
    Saves each Subset as a single .pt file containing stacked image and
    label tensors, matching the format the FGSM step below expects
    ("images", "labels" keys).
    """
    for i, part in enumerate(test_parts, start=1):
        images = torch.stack([part[j][0] for j in range(len(part))])
        labels = torch.tensor([part[j][1] for j in range(len(part))], dtype=torch.long)

        save_path = splits_dir / f"{prefix}_test_part_{i}.pt"
        torch.save({"images": images, "labels": labels}, save_path)
        print(f"  Saved: {save_path} ({images.shape[0]} samples)")


# =============================================================================
# FGSM ADVERSARIAL GENERATION
# =============================================================================

def fgsm_attack(
    model: nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    epsilon: float,
    mean: tuple[float, ...],
    std: tuple[float, ...],
) -> torch.Tensor:
    """
    Generates FGSM adversarial examples.

    Perturbation happens in raw pixel space [0, 1] (so the saved images
    stay valid, viewable images), but the gradient is computed through the
    model's expected normalized input -- i.e. we perturb the same tensor
    the model would see, just before normalization is applied for the
    forward pass, so `images.grad` reflects how each RAW pixel affects the
    model's loss.
    """
    images = images.clone().detach().to(DEVICE).requires_grad_(True)
    labels = labels.clone().detach().to(DEVICE)

    normalize = transforms.Normalize(mean, std)
    outputs = model(normalize(images))
    loss = F.cross_entropy(outputs, labels)

    model.zero_grad()
    loss.backward()

    grad_sign = images.grad.sign()
    perturbed = images + epsilon * grad_sign
    perturbed = torch.clamp(perturbed, 0.0, 1.0)

    return perturbed.detach().cpu()


def generate_adversarial_splits(
    model: nn.Module,
    splits_dir: Path,
    prefix: str,
    mean: tuple[float, ...],
    std: tuple[float, ...],
    dataset_name: str,
) -> None:
    """
    For each of the NUM_PARTS saved splits, generates FGSM adversarial
    examples at an increasing epsilon (part 1 -> eps=0.1, ..., part 10 ->
    eps=1.0) and saves the perturbed images alongside the originals.
    """
    for i in tqdm(
        range(1, NUM_PARTS + 1),
        desc=f"{dataset_name}: generating FGSM adversarial splits",
    ):
        split_path = splits_dir / f"{prefix}_test_part_{i}.pt"
        data = torch.load(split_path)

        images = data["images"]
        labels = data["labels"]
        epsilon = i / 10

        perturbed_images = fgsm_attack(model, images, labels, epsilon, mean, std)

        output_path = splits_dir / f"{prefix}_test_part_{i}_fgsm_eps_{epsilon:.1f}.pt"
        torch.save(
            {
                "images": perturbed_images,
                "labels": labels,
                "epsilon": epsilon,
                "attack": "fgsm",
                "dataset": dataset_name,
            },
            output_path,
        )
        print(f"  Saved perturbed dataset: {output_path}")


# =============================================================================
# QUICK ROBUSTNESS CHECK (optional but useful sanity check)
# =============================================================================

def evaluate_accuracy(
    model: nn.Module,
    images: torch.Tensor,
    labels: torch.Tensor,
    mean: tuple[float, ...],
    std: tuple[float, ...],
) -> float:
    normalize = transforms.Normalize(mean, std)
    with torch.no_grad():
        outputs = model(normalize(images.to(DEVICE)))
        predictions = torch.argmax(outputs, dim=1).cpu()
    accuracy = (predictions == labels).float().mean().item()
    return accuracy


def report_robustness(
    model: nn.Module,
    splits_dir: Path,
    prefix: str,
    mean: tuple[float, ...],
    std: tuple[float, ...],
    dataset_name: str,
) -> None:
    print("\n" + "=" * 60)
    print(f"{dataset_name}: Clean vs. FGSM-perturbed accuracy per epsilon")
    print("=" * 60)

    for i in range(1, NUM_PARTS + 1):
        epsilon = i / 10

        clean_data = torch.load(splits_dir / f"{prefix}_test_part_{i}.pt")
        adv_data = torch.load(splits_dir / f"{prefix}_test_part_{i}_fgsm_eps_{epsilon:.1f}.pt")

        clean_acc = evaluate_accuracy(model, clean_data["images"], clean_data["labels"], mean, std)
        adv_acc = evaluate_accuracy(model, adv_data["images"], adv_data["labels"], mean, std)

        print(
            f"  Part {i:2d}  eps={epsilon:.1f}  "
            f"clean_acc={clean_acc:.4f}  adv_acc={adv_acc:.4f}  "
            f"drop={clean_acc - adv_acc:+.4f}"
        )


# =============================================================================
# PER-DATASET PIPELINE
# =============================================================================

def run_for_dataset(dataset_name: str, config: dict) -> None:
    print("\n" + "=" * 78)
    print(f"DATASET: {dataset_name}")
    print("=" * 78)

    splits_dir = config["splits_dir"]
    prefix = dataset_name.lower()
    mean = config["mean"]
    std = config["std"]

    print("Loading and splitting test set...")
    test_parts = load_and_split_test_set(config["dataset_class"], dataset_name)

    print("\nSaving splits to disk...")
    save_splits_to_disk(test_parts, splits_dir, prefix)

    print(f"\nLoading model checkpoint: {config['checkpoint_path']}")
    model = load_model(config["checkpoint_path"])

    print("\nGenerating FGSM adversarial examples per split...")
    generate_adversarial_splits(model, splits_dir, prefix, mean, std, dataset_name)

    report_robustness(model, splits_dir, prefix, mean, std, dataset_name)


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    for dataset_name, config in DATASET_CONFIGS.items():
        try:
            run_for_dataset(dataset_name, config)
        except FileNotFoundError as error:
            print(
                f"\n[{dataset_name} SKIPPED] Missing checkpoint or data: {error}\n"
                f"Make sure '{config['checkpoint_path']}' exists in the working "
                f"directory before running this script."
            )

    print("\nDone.")


if __name__ == "__main__":
    main()



DATASET: MNIST
Loading and splitting test set...

MNIST: 10,000 total test samples
  Part 1: 1000 samples
  Part 2: 1000 samples
  Part 3: 1000 samples
  Part 4: 1000 samples
  Part 5: 1000 samples
  Part 6: 1000 samples
  Part 7: 1000 samples
  Part 8: 1000 samples
  Part 9: 1000 samples
  Part 10: 1000 samples

Saving splits to disk...
  Saved: mnist_test_splits\mnist_test_part_1.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_2.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_3.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_4.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_5.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_6.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_7.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_8.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_9.pt (1000 samples)
  Saved: mnist_test_splits\mnist_test_part_10.pt (1000 samples)

Loading model checkpoint: m

MNIST: generating FGSM adversarial splits:   0%|          | 0/10 [00:00<?, ?it/s]

  Saved perturbed dataset: mnist_test_splits\mnist_test_part_1_fgsm_eps_0.1.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_2_fgsm_eps_0.2.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_3_fgsm_eps_0.3.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_4_fgsm_eps_0.4.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_5_fgsm_eps_0.5.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_6_fgsm_eps_0.6.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_7_fgsm_eps_0.7.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_8_fgsm_eps_0.8.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_9_fgsm_eps_0.9.pt
  Saved perturbed dataset: mnist_test_splits\mnist_test_part_10_fgsm_eps_1.0.pt

MNIST: Clean vs. FGSM-perturbed accuracy per epsilon
  Part  1  eps=0.1  clean_acc=0.9940  adv_acc=0.8520  drop=+0.1420
  Part  2  eps=0.2  clean_acc=0.9960  adv_acc=0.4960  drop=+0.5000
  Part  3  eps=0.3  c

FashionMNIST: generating FGSM adversarial splits:   0%|          | 0/10 [00:00<?, ?it/s]

  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_1_fgsm_eps_0.1.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_2_fgsm_eps_0.2.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_3_fgsm_eps_0.3.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_4_fgsm_eps_0.4.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_5_fgsm_eps_0.5.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_6_fgsm_eps_0.6.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_7_fgsm_eps_0.7.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_8_fgsm_eps_0.8.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_9_fgsm_eps_0.9.pt
  Saved perturbed dataset: fashionmnist_test_splits\fashionmnist_test_part_10_fgsm_eps_1.0.pt

FashionMNIST: Clean vs. FGSM-perturbed accuracy per epsilon
  Part  

KMNIST: generating FGSM adversarial splits:   0%|          | 0/10 [00:00<?, ?it/s]

  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_1_fgsm_eps_0.1.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_2_fgsm_eps_0.2.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_3_fgsm_eps_0.3.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_4_fgsm_eps_0.4.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_5_fgsm_eps_0.5.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_6_fgsm_eps_0.6.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_7_fgsm_eps_0.7.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_8_fgsm_eps_0.8.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_9_fgsm_eps_0.9.pt
  Saved perturbed dataset: kmnist_test_splits\kmnist_test_part_10_fgsm_eps_1.0.pt

KMNIST: Clean vs. FGSM-perturbed accuracy per epsilon
  Part  1  eps=0.1  clean_acc=0.9440  adv_acc=0.5440  drop=+0.4000
  Part  2  eps=0.2  clean_acc=0.9370  adv_acc=0.2740  drop=+0.6630
